# BƯỚC 5: End-to-End Pipeline & CHUE Soft Voting

**Mục đích:** Hội tụ 2 luồng A (EfficientNetV2 từ Bước 4) và B (YOLO+BoT-SORT từ Bước 3) để ra kết luận cuối cùng cho từng xe trên tập **test**.

**Luồng xử lý:**
```
Test clips → YOLO+BoT-SORT → crops theo Track ID
                                    ↓ (khi track chết)
                        EfficientNetV2 (Bước 4)
                                    ↓
                    CHUE Soft Voting (Average Prob Pooling)
                                    ↓
              vector 6-node nhị phân + violation_flag
```

**Input:**
- YOLO weights: `/kaggle/input/datasets/truonglongty/yolo-best/yolo_best.pt`
- EfficientNetV2 weights: `/kaggle/input/datasets/truonglongty/efficientnet-best/efficientnet_best.pth`
- Test clips: từ `data_split.csv` (Set == 'test')

**Output:**
- `/kaggle/working/step5_results/step5_predictions.csv`
- `/kaggle/working/step5_results/step5_summary.json`

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kronomy/helmet-dataset-by-osf-lite")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite


## Cell 1: Cài đặt thư viện

In [2]:
# Cài đặt các thư viện cần thiết (chỉ chạy lần đầu)
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

pip_install('timm')
print('✅ Thư viện đã sẵn sàng')

✅ Thư viện đã sẵn sàng


In [3]:
pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.0 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


## Cell 2: Import & Cấu hình

In [4]:
import os, json, time, re, shutil, warnings
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# ── Đường dẫn ──────────────────────────────────────────────────────────────
BASE_DATASET = Path('/kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset')
SPLIT_CSV    = BASE_DATASET / 'data_split.csv'
IMAGES_DIR   = BASE_DATASET / 'images'
ANNOT_DIR    = BASE_DATASET / 'annotation' / 'annotation'

YOLO_WEIGHTS = Path('/kaggle/input/datasets/uynguyenthai/yolo11n-weight/best.pt')
EFFNET_CKPT  = Path('/kaggle/input/datasets/uynguyenthai/classifier-weight/efficientnet_best.pth')

OUT_DIR      = Path('/kaggle/working/step5_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Tham số ────────────────────────────────────────────────────────────────
CONF_THRESH       = 0.25   # YOLO confidence threshold
IOU_THRESH        = 0.45   # YOLO NMS IoU
TRACK_BUFFER      = 10     # frame giữ track khi bị khuất (10fps = 1s)
MAX_CROPS         = 80     # giới hạn crops/track để tránh OOM
CROP_PADDING      = 0.05   # padding 5% xung quanh bbox
INPUT_SIZE        = 224    # kích thước input EfficientNetV2
BATCH_SIZE        = 32     # batch size khi classify
NUM_WORKERS       = 2

# CHUE Soft Voting thresholds (per-node)
CHUE_THRESHOLDS   = [0.5, 0.5, 0.3, 0.3, 0.2, 0.2]
NODE_NAMES        = ['D_H', 'D_NH', 'P1_H', 'P1_NH', 'Extra_H', 'Extra_NH']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
print(f'📂 Output: {OUT_DIR}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🖥️  Device: cuda
📂 Output: /kaggle/working/step5_results


## Cell 3: Kiểm tra file weights

In [5]:
def check_file(path: Path, name: str):
    """Kiểm tra file tồn tại, in kích thước."""
    if not path.exists():
        raise FileNotFoundError(f'❌ Không tìm thấy {name}: {path}')
    size_mb = path.stat().st_size / 1e6
    print(f'✅ {name}: {path.name}  ({size_mb:.1f} MB)')

check_file(YOLO_WEIGHTS, 'YOLO weights')
check_file(EFFNET_CKPT,  'EfficientNetV2 checkpoint')
print(f'✅ Dataset: {IMAGES_DIR}')
print(f'✅ Split CSV: {SPLIT_CSV}')

✅ YOLO weights: best.pt  (5.4 MB)
✅ EfficientNetV2 checkpoint: efficientnet_best.pth  (243.5 MB)
✅ Dataset: /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset/images
✅ Split CSV: /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset/data_split.csv


## Cell 4: Load danh sách test clips

In [6]:
def load_test_clips(split_csv: Path) -> List[str]:
    """
    Đọc data_split.csv và trả về danh sách tên clip thuộc tập test.
    Cột 'Set' có giá trị: training / validation / test
    """
    df = pd.read_csv(split_csv)
    # Tên cột có thể là 'Set', 'split', 'set' — tự detect
    col_candidates = [c for c in df.columns if c.lower() in ('set', 'split')]
    if not col_candidates:
        raise ValueError(f'Không tìm thấy cột split trong {split_csv}. Cột hiện có: {df.columns.tolist()}')
    split_col = col_candidates[0]
    clip_col_candidates = [c for c in df.columns if 'clip' in c.lower() or 'video' in c.lower() or 'name' in c.lower()]
    clip_col = clip_col_candidates[0] if clip_col_candidates else df.columns[0]
    
    test_clips = df[df[split_col] == 'test'][clip_col].tolist()
    print(f'📋 Tổng test clips: {len(test_clips)}')
    return test_clips

def find_clip_dir(clip_name: str, images_dir: Path) -> Optional[Path]:
    """
    Tìm thư mục chứa frame của clip_name trong images/part_X/{clip_name}/.
    """
    for part_dir in sorted(images_dir.iterdir()):
        if not part_dir.is_dir():
            continue
        clip_dir = part_dir / clip_name
        if clip_dir.exists():
            return clip_dir
    return None

TEST_CLIPS = load_test_clips(SPLIT_CSV)
# Kiểm tra 3 clip đầu
for c in TEST_CLIPS[:3]:
    d = find_clip_dir(c, IMAGES_DIR)
    print(f'  {c} → {d}')

📋 Tổng test clips: 182
  Bago_highway_10 → /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset/images/part_1/Bago_highway_10
  Bago_highway_15 → /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset/images/part_1/Bago_highway_15
  Bago_highway_20 → /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset/images/part_1/Bago_highway_20


## Cell 5: Tạo file cấu hình BoT-SORT

In [7]:
BOTSORT_CFG_PATH = Path('/kaggle/working/botsort_step5.yaml')

# FIX: Thêm ĐẦY ĐỦ tất cả params cho BoT-SORT từ phiên bản mới ultralytics
# Từ: https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/trackers/botsort.yaml
botsort_config = """tracker_type: botsort
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.25
track_buffer: 30
match_thresh: 0.8
fuse_score: True
gmc_method: sparseOptFlow
proximity_thresh: 0.5
appearance_thresh: 0.8
with_reid: False
model: auto
"""

BOTSORT_CFG_PATH.write_text(botsort_config)
print(f'✅ BoT-SORT config: {BOTSORT_CFG_PATH}')
print('   Đã thêm đầy đủ tất cả parameters từ phiên bản mới ultralytics:')
print('   - proximity_thresh: 0.5')
print('   - appearance_thresh: 0.8')
print('   - track_buffer: 30 (tăng từ 10 để tái gắn kết tốt hơn)')


✅ BoT-SORT config: /kaggle/working/botsort_step5.yaml
   Đã thêm đầy đủ tất cả parameters từ phiên bản mới ultralytics:
   - proximity_thresh: 0.5
   - appearance_thresh: 0.8
   - track_buffer: 30 (tăng từ 10 để tái gắn kết tốt hơn)


## Cell 6: TrackManager — quản lý vòng đời track & crops

In [8]:
class TrackManager:
    """
    Quản lý vòng đời từng Track ID trong 1 video clip.
    - Tích lũy crops cho mỗi track (tối đa MAX_CROPS)
    - Khi track 'chết' (không xuất hiện > TRACK_BUFFER frames): 
      flush crops vào memory để classifier đọc sau
    - Hỗ trợ finalize() sau khi hết frame
    """

    def __init__(self, clip_name: str, track_buffer: int = TRACK_BUFFER,
                 max_crops: int = MAX_CROPS, crop_padding: float = CROP_PADDING,
                 input_size: int = INPUT_SIZE):
        self.clip_name    = clip_name
        self.track_buffer = track_buffer
        self.max_crops    = max_crops
        self.crop_padding = crop_padding
        self.input_size   = input_size

        # dict: track_id → {'crops': [np.ndarray], 'last_seen': int, 'bbox_list': [x1y1x2y2]}
        self._active: Dict[int, dict] = {}
        # list of completed tracks: {'track_id', 'crops', 'first_frame', 'last_frame', 'bbox_list'}
        self.completed: List[dict] = []

    # ── internal ──────────────────────────────────────────────────────────
    def _crop_frame(self, frame: np.ndarray, x1: float, y1: float,
                    x2: float, y2: float) -> np.ndarray:
        """Crop vùng xe (có padding), resize về input_size."""
        H, W = frame.shape[:2]
        pw = (x2 - x1) * self.crop_padding
        ph = (y2 - y1) * self.crop_padding
        cx1 = max(0,   int(x1 - pw))
        cy1 = max(0,   int(y1 - ph))
        cx2 = min(W-1, int(x2 + pw))
        cy2 = min(H-1, int(y2 + ph))
        crop = frame[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            return None
        crop = cv2.resize(crop, (self.input_size, self.input_size))
        return crop  # BGR, uint8

    def _flush_track(self, track_id: int):
        """Đẩy track đã kết thúc sang list completed, xoá khỏi active."""
        info = self._active.pop(track_id)
        if len(info['crops']) == 0:
            return  # không có crop nào hợp lệ → bỏ qua
        self.completed.append({
            'track_id':    track_id,
            'crops':       info['crops'],       # list of np.ndarray (RGB)
            'first_frame': info['first_frame'],
            'last_frame':  info['last_seen'],
            'bbox_list':   info['bbox_list'],   # [(x1,y1,x2,y2), ...]
        })

    # ── public API ────────────────────────────────────────────────────────
    def update(self, frame_idx: int, frame: np.ndarray,
               detections: List[tuple]):
        """
        Cập nhật mỗi frame.
        detections: list of (track_id, x1, y1, x2, y2)
        """
        seen_ids = set()
        for (tid, x1, y1, x2, y2) in detections:
            tid = int(tid)
            seen_ids.add(tid)
            crop = self._crop_frame(frame, x1, y1, x2, y2)
            if crop is None:
                continue
            # Chuyển BGR→RGB để tương thích với PyTorch transforms
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

            if tid not in self._active:
                self._active[tid] = {
                    'crops':       [],
                    'first_frame': frame_idx,
                    'last_seen':   frame_idx,
                    'bbox_list':   [],
                }
            info = self._active[tid]
            info['last_seen'] = frame_idx
            if len(info['crops']) < self.max_crops:
                info['crops'].append(crop_rgb)
            info['bbox_list'].append((x1, y1, x2, y2))

        # Death check: flush các track không xuất hiện quá track_buffer frames
        dead_ids = [
            tid for tid, info in self._active.items()
            if (frame_idx - info['last_seen']) > self.track_buffer
        ]
        for tid in dead_ids:
            self._flush_track(tid)

    def finalize(self):
        """Flush tất cả tracks còn sót lại sau khi hết frame."""
        for tid in list(self._active.keys()):
            self._flush_track(tid)

print('✅ TrackManager đã sẵn sàng')

✅ TrackManager đã sẵn sàng


## Cell 7: EfficientNetV2 Classifier

In [9]:
def build_efficientnet(num_nodes: int = 6) -> nn.Module:
    """Xây dựng EfficientNetV2-S với classification head 6-node."""
    model = models.efficientnet_v2_s(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_nodes),
    )
    return model

def load_classifier(ckpt_path: Path, device: str) -> nn.Module:
    """Load EfficientNetV2 từ checkpoint Bước 4."""
    model = build_efficientnet(num_nodes=6)
    ckpt  = torch.load(ckpt_path, map_location=device)
    # Tương thích với các cách lưu khác nhau
    if isinstance(ckpt, dict):
        state = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt))
    else:
        state = ckpt
    model.load_state_dict(state, strict=True)
    model.to(device)
    model.eval()
    print(f'✅ EfficientNetV2 loaded từ {ckpt_path.name}')
    return model

# Transform cho inference (không augmentation)
INFER_TRANSFORM = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

CLASSIFIER = load_classifier(EFFNET_CKPT, DEVICE)


✅ EfficientNetV2 loaded từ efficientnet_best.pth


## Cell 8: CHUE Soft Voting

In [10]:
def chue_soft_voting(raw_probabilities: List[List[float]],
                     thresholds: List[float] = None) -> List[int]:
    """
    CHUE Soft Voting — Average Probability Pooling + Per-node Threshold.

    Args:
        raw_probabilities: List of 6-node sigmoid probabilities per frame
                           shape: (n_frames, 6)
        thresholds: Ngưỡng quyết định riêng cho từng node.
                    Default: [0.5, 0.5, 0.3, 0.3, 0.2, 0.2]

    Returns:
        vector 6-node nhị phân [0/1] — kết luận cuối cùng cho track
    """
    if thresholds is None:
        # Driver→0.5, Passenger1→0.3, Extra→0.2 (phù hợp class imbalance)
        thresholds = CHUE_THRESHOLDS

    if len(raw_probabilities) == 0:
        return [0] * 6

    probs = np.array(raw_probabilities, dtype=np.float32)  # (n_frames, 6)
    track_mean_probs = probs.mean(axis=0)                  # (6,) — trung bình theo thời gian
    final = (track_mean_probs >= np.array(thresholds)).astype(int)
    return final.tolist()


def vector_to_label_str(vec: List[int]) -> str:
    """
    Chuyển vector 6-node nhị phân sang chuỗi nhãn compound.
    vec = [D_H, D_NH, P1_H, P1_NH, Extra_H, Extra_NH]
    """
    parts = []
    if vec[0]: parts.append('DHelmet')
    if vec[1]: parts.append('DNoHelmet')
    if vec[2]: parts.append('P1Helmet')
    if vec[3]: parts.append('P1NoHelmet')
    if vec[4]: parts.append('ExtraHelmet')
    if vec[5]: parts.append('ExtraNoHelmet')
    return ''.join(parts) if parts else 'Unknown'


print('✅ CHUE Soft Voting functions sẵn sàng')
# ── Quick sanity test ──
_test_probs = [[0.9, 0.05, 0.6, 0.1, 0.05, 0.05],
               [0.8, 0.02, 0.7, 0.05, 0.1, 0.02]]
_res = chue_soft_voting(_test_probs)
print(f'   Sanity test → {_res} → "{vector_to_label_str(_res)}"')

✅ CHUE Soft Voting functions sẵn sàng
   Sanity test → [1, 0, 1, 0, 0, 0] → "DHelmetP1Helmet"


## Cell 9: Hàm classify một batch crops

In [11]:
class CropListDataset(Dataset):
    """Dataset in-memory từ list numpy arrays (RGB uint8)."""
    def __init__(self, crops: List[np.ndarray], transform):
        self.crops     = crops
        self.transform = transform

    def __len__(self):
        return len(self.crops)

    def __getitem__(self, idx):
        img = Image.fromarray(self.crops[idx])  # RGB PIL
        return self.transform(img)


@torch.no_grad()
def classify_crops(crops: List[np.ndarray],
                   model: nn.Module,
                   device: str,
                   batch_size: int = BATCH_SIZE) -> List[List[float]]:
    """
    Đẩy list crops qua EfficientNetV2, trả về list xác suất thô (sau Sigmoid).

    Args:
        crops: List of np.ndarray (H,W,3) RGB
    Returns:
        List of [p0..p5] float cho mỗi crop
    """
    if not crops:
        return []

    ds     = CropListDataset(crops, INFER_TRANSFORM)
    loader = DataLoader(ds, batch_size=batch_size,
                        shuffle=False, num_workers=0, pin_memory=(device=='cuda'))

    all_probs = []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)              # (B, 6)
        probs  = torch.sigmoid(logits)     # (B, 6)
        all_probs.extend(probs.cpu().numpy().tolist())

    return all_probs  # List[ [p0..p5] ]

print('✅ classify_crops() sẵn sàng')

✅ classify_crops() sẵn sàng


## Cell 10: Pipeline xử lý 1 clip

In [12]:
def process_clip(clip_name: str,
                 clip_dir: Path,
                 yolo_model: YOLO,
                 classifier: nn.Module,
                 device: str,
                 thresholds: List[float] = None) -> List[dict]:
    """
    Xử lý 1 clip: YOLO+BoT-SORT tracking → CHUE Voting → kết quả.

    Args:
        clip_name : tên clip (VD: 'clip_001')
        clip_dir  : Path đến thư mục chứa frames (1.jpg → 100.jpg)
        yolo_model: YOLO object đã load weights
        classifier: EfficientNetV2 đã load weights

    Returns:
        List of dict — mỗi phần tử là kết quả 1 track:
        {
          'clip_name', 'track_id_predicted', 'n_crops_voted',
          'label_vector', 'label_string', 'violation_flag',
          'mean_probs', 'first_frame', 'last_frame', 'avg_bbox'
        }
    """
    results = []

    # Lấy danh sách frame (sắp xếp số tự nhiên 1→100)
    frame_files = sorted(
        [f for f in clip_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')],
        key=lambda p: int(p.stem)
    )
    if not frame_files:
        print(f'  ⚠️  {clip_name}: không tìm thấy frame, bỏ qua')
        return results

    # Reset tracker cho clip mới (cách an toàn nhất)
    yolo_model.predictor = None

    manager = TrackManager(clip_name=clip_name)

    for frame_idx, frame_path in enumerate(frame_files):
        # Đọc frame
        frame = cv2.imread(str(frame_path))
        if frame is None:
            continue

        # YOLO + BoT-SORT inference
        try:
            track_results = yolo_model.track(
                source=frame,
                persist=True,
                conf=CONF_THRESH,
                iou=IOU_THRESH,
                tracker=str(BOTSORT_CFG_PATH),
                verbose=False,
            )
        except Exception as e:
            print(f'  ⚠️  {clip_name} frame {frame_idx}: YOLO lỗi — {e}')
            continue

        # Parse detections
        detections = []
        if track_results and track_results[0].boxes is not None:
            boxes = track_results[0].boxes
            if boxes.id is not None:
                for i, tid in enumerate(boxes.id.tolist()):
                    x1, y1, x2, y2 = boxes.xyxy[i].tolist()
                    detections.append((int(tid), x1, y1, x2, y2))

        # Cập nhật TrackManager
        manager.update(frame_idx, frame, detections)

        # Classify ngay các track đã completed để giải phóng RAM
        if manager.completed:
            for track_info in manager.completed:
                _classify_and_record(track_info, clip_name, classifier, device, thresholds, results)
            manager.completed.clear()

    # Finalize: flush các track còn sót
    manager.finalize()
    for track_info in manager.completed:
        _classify_and_record(track_info, clip_name, classifier, device, thresholds, results)
    manager.completed.clear()

    # Giải phóng bộ nhớ GPU
    torch.cuda.empty_cache() if device == 'cuda' else None

    return results


def _classify_and_record(track_info: dict, clip_name: str,
                          classifier: nn.Module, device: str,
                          thresholds: List[float], results: list):
    """Classify crops của 1 track và ghi vào results."""
    crops   = track_info['crops']
    tid     = track_info['track_id']

    # Classify
    raw_probs = classify_crops(crops, classifier, device)
    if not raw_probs:
        return

    # CHUE Soft Voting
    label_vec  = chue_soft_voting(raw_probs, thresholds)
    label_str  = vector_to_label_str(label_vec)
    vio_flag   = int(label_vec[1] or label_vec[3] or label_vec[5])  # bất kỳ NoHelmet
    mean_probs = np.array(raw_probs).mean(axis=0).tolist()

    # Tính avg bbox
    bboxes = track_info['bbox_list']
    avg_bbox = np.mean(bboxes, axis=0).tolist() if bboxes else [0,0,0,0]

    results.append({
        'clip_name':          clip_name,
        'track_id_predicted': tid,
        'first_frame':        track_info['first_frame'],
        'last_frame':         track_info['last_frame'],
        'n_crops_voted':      len(crops),
        'label_vector':       label_vec,      # [0/1] x6
        'label_string':       label_str,
        'violation_flag':     vio_flag,
        'mean_probs':         [round(p,4) for p in mean_probs],
        'avg_bbox':           [round(v,1) for v in avg_bbox],
        **{f'lv_{n}': v for n, v in zip(NODE_NAMES, label_vec)},
        **{f'mp_{n}': round(p,4) for n, p in zip(NODE_NAMES, mean_probs)},
    })

print('✅ process_clip() sẵn sàng')

✅ process_clip() sẵn sàng


## Cell 11: Chạy pipeline toàn bộ tập test

In [13]:
# Load YOLO model (1 lần duy nhất)
print('🔄 Đang load YOLO model...')
yolo_model = YOLO(str(YOLO_WEIGHTS))
print(f'✅ YOLO loaded: {YOLO_WEIGHTS.name}')

# ── Chạy pipeline ──────────────────────────────────────────────────────────
all_results   = []
failed_clips  = []
t_start       = time.time()

print(f'\n🚀 Bắt đầu pipeline — {len(TEST_CLIPS)} clips\n')
print(f'   CHUE thresholds: {CHUE_THRESHOLDS}')
print('-' * 60)

for i, clip_name in enumerate(TEST_CLIPS):
    clip_dir = find_clip_dir(clip_name, IMAGES_DIR)
    if clip_dir is None:
        print(f'[{i+1:3d}/{len(TEST_CLIPS)}] ⚠️  {clip_name}: không tìm thấy thư mục')
        failed_clips.append({'clip_name': clip_name, 'reason': 'dir_not_found'})
        continue

    t0 = time.time()
    try:
        clip_results = process_clip(
            clip_name=clip_name,
            clip_dir=clip_dir,
            yolo_model=yolo_model,
            classifier=CLASSIFIER,
            device=DEVICE,
            thresholds=CHUE_THRESHOLDS,
        )
        all_results.extend(clip_results)
        elapsed = time.time() - t0
        n_tracks = len(clip_results)
        n_vio    = sum(r['violation_flag'] for r in clip_results)
        print(f'[{i+1:3d}/{len(TEST_CLIPS)}] ✅ {clip_name:30s} '
              f'tracks={n_tracks:3d}  viol={n_vio:3d}  {elapsed:.1f}s')

    except Exception as e:
        print(f'[{i+1:3d}/{len(TEST_CLIPS)}] ❌ {clip_name}: {e}')
        failed_clips.append({'clip_name': clip_name, 'reason': str(e)})
        # Reset model & giải phóng RAM để tiếp tục
        try:
            yolo_model.predictor = None
            torch.cuda.empty_cache()
        except:
            pass

total_time = time.time() - t_start
print('\n' + '=' * 60)
print(f'🏁 Xong! Tổng thời gian: {total_time/60:.1f} phút')
print(f'   Tổng tracks: {len(all_results)}')
print(f'   Clips thất bại: {len(failed_clips)}')

🔄 Đang load YOLO model...
✅ YOLO loaded: best.pt

🚀 Bắt đầu pipeline — 182 clips

   CHUE thresholds: [0.5, 0.5, 0.3, 0.3, 0.2, 0.2]
------------------------------------------------------------
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 274ms
Prepared 1 package in 105ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

[  1/182] ✅ Bago_highway_10                tracks=  9  viol=  6  11.4s
[  2/182] ✅ Bago_highway_15                tracks=  4  viol=  3  8.7s
[  3/182] ✅ Bago_highway_20                tracks=  7  viol=  1  11.5s
[  4/182] ✅ Bago_highway_26                tracks=  6  viol=  2  13.7s
[  5/182] ✅ Bago_highway_29                tracks= 11  viol=  3  16.2s
[  6/182] ✅ Bago_highway_31                tracks=  1  viol=  0  17.5s
[  7/182] ✅ Bago_highwa

## Cell 12: Lưu kết quả ra CSV

In [14]:
if not all_results:
    print('⚠️  Không có kết quả nào để lưu!')
else:
    df_pred = pd.DataFrame(all_results)

    # ── Sắp xếp cột ──
    front_cols = [
        'clip_name', 'track_id_predicted', 'first_frame', 'last_frame',
        'n_crops_voted', 'label_string', 'violation_flag',
    ]
    node_label_cols = [f'lv_{n}' for n in NODE_NAMES]
    node_prob_cols  = [f'mp_{n}' for n in NODE_NAMES]
    other_cols = ['label_vector', 'mean_probs', 'avg_bbox']
    ordered = front_cols + node_label_cols + node_prob_cols + other_cols
    df_pred = df_pred[[c for c in ordered if c in df_pred.columns]]

    out_csv = OUT_DIR / 'step5_predictions.csv'
    df_pred.to_csv(out_csv, index=False)
    print(f'✅ Đã lưu predictions: {out_csv}')
    print(f'   Shape: {df_pred.shape}')

    # Xem mẫu
    display(df_pred[front_cols + node_label_cols].head(10))

✅ Đã lưu predictions: /kaggle/working/step5_results/step5_predictions.csv
   Shape: (435, 22)


,clip_name,track_id_predicted,first_frame,last_frame,n_crops_voted,label_string,violation_flag,lv_D_H,lv_D_NH,lv_P1_H,lv_P1_NH,lv_Extra_H,lv_Extra_NH
0,Bago_highway_10,3,0,6,7,DHelmetP1NoHelmetExtraHelmetExtraNoHelmet,1,1,0,0,1,1,1
1,Bago_highway_10,12,50,63,14,DHelmet,0,1,0,0,0,0,0
2,Bago_highway_10,14,53,63,10,DNoHelmet,1,0,1,0,0,0,0
3,Bago_highway_10,36,70,79,10,DHelmet,0,1,0,0,0,0,0
4,Bago_highway_10,1,0,99,80,DNoHelmet,1,0,1,0,0,0,0
5,Bago_highway_10,2,0,99,80,DNoHelmet,1,0,1,0,0,0,0
6,Bago_highway_10,7,40,96,55,DHelmet,0,1,0,0,0,0,0
7,Bago_highway_10,38,74,95,22,DHelmetP1NoHelmet,1,1,0,0,1,0,0
8,Bago_highway_10,48,95,99,5,DNoHelmetP1NoHelmet,1,0,1,0,1,0,0
9,Bago_highway_15,1,0,18,19,DHelmet,0,1,0,0,0,0,0


## Cell 13: Thống kê & lưu summary JSON

In [15]:
if all_results:
    # ── Phân phối violation ──
    n_total   = len(df_pred)
    n_viol    = df_pred['violation_flag'].sum()
    n_compl   = n_total - n_viol
    n_clips   = df_pred['clip_name'].nunique()

    # Phân phối label_string
    label_dist = df_pred['label_string'].value_counts().head(15).to_dict()

    # Tỷ lệ dương per node
    node_pos_rate = {
        n: round(df_pred[f'lv_{n}'].mean(), 4)
        for n in NODE_NAMES if f'lv_{n}' in df_pred.columns
    }

    summary = {
        'step': 5,
        'description': 'End-to-End Pipeline + CHUE Soft Voting',
        'total_test_clips':  len(TEST_CLIPS),
        'processed_clips':   n_clips,
        'failed_clips':      len(failed_clips),
        'total_tracks':      n_total,
        'violation_tracks':  int(n_viol),
        'compliant_tracks':  int(n_compl),
        'violation_rate':    round(n_viol / max(n_total,1), 4),
        'chue_thresholds':   CHUE_THRESHOLDS,
        'node_names':        NODE_NAMES,
        'node_positive_rate': node_pos_rate,
        'top_labels':        label_dist,
        'failed_clip_list':  failed_clips,
        'total_time_sec':    round(total_time, 1),
        'output_csv':        str(out_csv),
    }

    out_json = OUT_DIR / 'step5_summary.json'
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print('=' * 60)
    print('📊 TỔNG KẾT BƯỚC 5')
    print('=' * 60)
    print(f'   Tổng test clips     : {len(TEST_CLIPS)}')
    print(f'   Clips xử lý thành công: {n_clips}')
    print(f'   Tổng tracks phán đoán: {n_total}')
    print(f'   Violation  : {n_viol} ({n_viol/max(n_total,1)*100:.1f}%)')
    print(f'   Compliant  : {n_compl} ({n_compl/max(n_total,1)*100:.1f}%)')
    print(f'\n   [Per-node positive rate]')
    for n, r in node_pos_rate.items():
        print(f'   {n:15s}: {r:.3f}  ({r*100:.1f}%)')
    print(f'\n✅ Summary: {out_json}')

📊 TỔNG KẾT BƯỚC 5
   Tổng test clips     : 182
   Clips xử lý thành công: 56
   Tổng tracks phán đoán: 435
   Violation  : 148 (34.0%)
   Compliant  : 287 (66.0%)

   [Per-node positive rate]
   D_H            : 0.729  (72.9%)
   D_NH           : 0.276  (27.6%)
   P1_H           : 0.324  (32.4%)
   P1_NH          : 0.177  (17.7%)
   Extra_H        : 0.051  (5.1%)
   Extra_NH       : 0.108  (10.8%)

✅ Summary: /kaggle/working/step5_results/step5_summary.json


## Cell 14: Kiểm tra nhanh — xem mẫu kết quả chi tiết

In [16]:
if all_results:
    print('📋 Mẫu 5 tracks vi phạm:')
    viol_df = df_pred[df_pred['violation_flag'] == 1]
    display(viol_df[front_cols + node_label_cols].head(5))

    print('\n📋 Mẫu 5 tracks tuân thủ:')
    compl_df = df_pred[df_pred['violation_flag'] == 0]
    display(compl_df[front_cols + node_label_cols].head(5))

    print('\n📊 Phân phối số crops/track:')
    print(df_pred['n_crops_voted'].describe().round(1))

📋 Mẫu 5 tracks vi phạm:


,clip_name,track_id_predicted,first_frame,last_frame,n_crops_voted,label_string,violation_flag,lv_D_H,lv_D_NH,lv_P1_H,lv_P1_NH,lv_Extra_H,lv_Extra_NH
0,Bago_highway_10,3,0,6,7,DHelmetP1NoHelmetExtraHelmetExtraNoHelmet,1,1,0,0,1,1,1
2,Bago_highway_10,14,53,63,10,DNoHelmet,1,0,1,0,0,0,0
4,Bago_highway_10,1,0,99,80,DNoHelmet,1,0,1,0,0,0,0
5,Bago_highway_10,2,0,99,80,DNoHelmet,1,0,1,0,0,0,0
7,Bago_highway_10,38,74,95,22,DHelmetP1NoHelmet,1,1,0,0,1,0,0



📋 Mẫu 5 tracks tuân thủ:


,clip_name,track_id_predicted,first_frame,last_frame,n_crops_voted,label_string,violation_flag,lv_D_H,lv_D_NH,lv_P1_H,lv_P1_NH,lv_Extra_H,lv_Extra_NH
1,Bago_highway_10,12,50,63,14,DHelmet,0,1,0,0,0,0,0
3,Bago_highway_10,36,70,79,10,DHelmet,0,1,0,0,0,0,0
6,Bago_highway_10,7,40,96,55,DHelmet,0,1,0,0,0,0,0
9,Bago_highway_15,1,0,18,19,DHelmet,0,1,0,0,0,0,0
13,Bago_highway_20,1,0,2,3,DHelmet,0,1,0,0,0,0,0



📊 Phân phối số crops/track:
count    435.0
mean      20.4
std       20.9
min        1.0
25%        3.0
50%       14.0
75%       29.0
max       80.0
Name: n_crops_voted, dtype: float64


## Cell 15: (Tuỳ chọn) Tune CHUE thresholds và xem thay đổi

In [17]:
if all_results:
    print('🔧 So sánh CHUE thresholds:\n')
    threshold_configs = {
        'strict (0.5 all)':    [0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'default (D=0.5, P1=0.3, Extra=0.2)': [0.5, 0.5, 0.3, 0.3, 0.2, 0.2],
        'lenient (D=0.4, P1=0.25, Extra=0.15)': [0.4, 0.4, 0.25, 0.25, 0.15, 0.15],
    }

    for cfg_name, t in threshold_configs.items():
        # Recompute label_vec từ mean_probs đã lưu
        def recompute_vio(row, thresholds):
            mp = [row[f'mp_{n}'] for n in NODE_NAMES]
            vec = (np.array(mp) >= np.array(thresholds)).astype(int).tolist()
            return int(vec[1] or vec[3] or vec[5])

        df_pred[f'vio_{cfg_name}'] = df_pred.apply(lambda r: recompute_vio(r, t), axis=1)
        n_v = df_pred[f'vio_{cfg_name}'].sum()
        print(f'  [{cfg_name:45s}] violation={n_v:4d} ({n_v/len(df_pred)*100:.1f}%)')

    print('\n💡 Chọn threshold phù hợp trước khi chạy Bước 6 Evaluation.')

🔧 So sánh CHUE thresholds:

  [strict (0.5 all)                             ] violation= 131 (30.1%)
  [default (D=0.5, P1=0.3, Extra=0.2)           ] violation= 148 (34.0%)
  [lenient (D=0.4, P1=0.25, Extra=0.15)         ] violation= 169 (38.9%)

💡 Chọn threshold phù hợp trước khi chạy Bước 6 Evaluation.


## Cell 16: Verify output cho Bước 6

In [18]:
print('🔍 Verify output cho Bước 6...')
issues = []

# Kiểm tra file tồn tại
required_files = [
    OUT_DIR / 'step5_predictions.csv',
    OUT_DIR / 'step5_summary.json',
]
for f in required_files:
    if f.exists():
        print(f'  ✅ {f.name}  ({f.stat().st_size/1024:.1f} KB)')
    else:
        issues.append(f'Thiếu file: {f}')
        print(f'  ❌ {f.name}  — THIẾU!')

if all_results:
    # Kiểm tra cột bắt buộc cho Bước 6
    required_cols = ['clip_name', 'track_id_predicted', 'avg_bbox',
                     'violation_flag', 'label_vector'] + [f'lv_{n}' for n in NODE_NAMES]
    df_check = pd.read_csv(OUT_DIR / 'step5_predictions.csv')
    missing_cols = [c for c in required_cols if c not in df_check.columns]
    if missing_cols:
        issues.append(f'Thiếu cột: {missing_cols}')
        print(f'  ❌ Thiếu cột: {missing_cols}')
    else:
        print(f'  ✅ Tất cả cột bắt buộc cho Bước 6 đều có')

    # Kiểm tra không có Unknown quá nhiều
    n_unk = (df_check['label_string'] == 'Unknown').sum()
    if n_unk > len(df_check) * 0.3:
        issues.append(f'Quá nhiều Unknown ({n_unk}/{len(df_check)})')
        print(f'  ⚠️  Unknown nhiều ({n_unk}): EfficientNetV2 có thể chưa hội tụ tốt')
    else:
        print(f'  ✅ Unknown rate: {n_unk}/{len(df_check)} ({n_unk/len(df_check)*100:.1f}%)')

if issues:
    
    print(f'\n❌ Phát hiện {len(issues)} vấn đề:')
    for iss in issues:
        print(f'   • {iss}')
else:
    print('\n✅ Bước 5 HOÀN THÀNH — Sẵn sàng cho Bước 6 Evaluation!')

🔍 Verify output cho Bước 6...
  ✅ step5_predictions.csv  (84.5 KB)
  ✅ step5_summary.json  (11.4 KB)
  ✅ Tất cả cột bắt buộc cho Bước 6 đều có
  ✅ Unknown rate: 0/435 (0.0%)

✅ Bước 5 HOÀN THÀNH — Sẵn sàng cho Bước 6 Evaluation!


In [19]:
## Cell 14: Zip kết quả Bước 5
# Chạy cell này để nén kết quả Bước 5 lại thành file ZIP để dễ dàng tải về hoặc đưa vào input cho Bước 6
import shutil
import os

out_dir = '/kaggle/working/step5_results'
zip_path = '/kaggle/working/step5_results'

if os.path.exists(out_dir):
    shutil.make_archive(zip_path, 'zip', out_dir)
    print(f'✅ Đã nén thành công thư mục {out_dir}')
    print(f'   File zip tự động lưu tại: {zip_path}.zip')
    print('   Bạn có thể xem ở mục Output bên phải màn hình Kaggle để tải về, hoặc add Notebook này vào Data của Bước 6.')
else:
    print(f'❌ Không tìm thấy thư mục {out_dir}. Hãy đảm bảo Pipeline phía trên đã được chạy xong trước đó!')


✅ Đã nén thành công thư mục /kaggle/working/step5_results
   File zip tự động lưu tại: /kaggle/working/step5_results.zip
   Bạn có thể xem ở mục Output bên phải màn hình Kaggle để tải về, hoặc add Notebook này vào Data của Bước 6.
